# Data Audit and Cleaning

This notebook audits the raw JNK3/GSK3β docking dataset and prepares a cleaned molecular dataset for modelling. It checks missing values, duplicate SMILES and score distributions, canonicalises molecular structures, removes invalid or incomplete records, aggregates duplicate molecules, and creates binary activity labels using the docking-score threshold of -8.0. The cleaned dataset is saved for the subsequent scaffold-splitting and descriptor-generation steps.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from rdkit import Chem

DATA_FILE = "case_jnk3-gsk3b_docking_scores.csv"

# Docking-score cutoff 
DOCKING_CUTOFF = -8.0

# Random seed makes later results reproducible
RANDOM_SEED = 42

print("Libraries imported successfully.")
print("Docking cutoff:", DOCKING_CUTOFF)
print("Random seed:", RANDOM_SEED)

In [ ]:
raw_df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Number of rows:", raw_df.shape[0])
print("Number of columns:", raw_df.shape[1])

raw_df.head()

In [ ]:
print("DATASET SHAPE")
print(raw_df.shape)

print("\nCOLUMN NAMES")
print(raw_df.columns.tolist())

print("\nDATA TYPES")
print(raw_df.dtypes)

print("\nMISSING VALUES")
print(raw_df.isna().sum())

print("\nEXACT DUPLICATE SMILES")
print(raw_df.duplicated(subset="smiles").sum())

print("\nDOCKING SCORE SUMMARY")
display(
    raw_df[["gsk3b_score", "jnk3_score"]].describe()
)

In [ ]:
# Remove rows with missing SMILES and docking scores
missing_rows = raw_df[
    ["smiles", "gsk3b_score", "jnk3_score"]
].isna().any(axis=1).sum()

working_df = raw_df.dropna(
    subset=["smiles", "gsk3b_score", "jnk3_score"]
).copy()


#  convert SMILES into canonical SMILES
def canonicalise_smiles(smiles):
    molecule = Chem.MolFromSmiles(smiles)

    if molecule is None:
        return None

    return Chem.MolToSmiles(molecule)


# Create canonical SMILES
working_df["canonical_smiles"] = working_df["smiles"].apply(
    canonicalise_smiles
)

# remove invalid SMILES
invalid_smiles = working_df["canonical_smiles"].isna().sum()

working_df = working_df.dropna(
    subset=["canonical_smiles"]
).copy()


# remove rows where docking score  zero
zero_score_rows = (
    (working_df["gsk3b_score"] == 0) |
    (working_df["jnk3_score"] == 0)
).sum()

working_df = working_df[
    (working_df["gsk3b_score"] != 0) &
    (working_df["jnk3_score"] != 0)
].copy()

working_df = working_df.reset_index(drop=True)

print("Original rows:", len(raw_df))
print("Rows removed because of missing values:", missing_rows)
print("Invalid SMILES removed:", invalid_smiles)
print("Zero-score rows removed:", zero_score_rows)
print("Rows remaining:", len(working_df))

In [ ]:
# Combine duplicate molecules
rows_before_duplicates = len(working_df)

clean_df = (
    working_df
    .groupby("canonical_smiles", as_index=False)
    .agg(
        smiles=("smiles", "first"),
        gsk3b_score=("gsk3b_score", "mean"),
        jnk3_score=("jnk3_score", "mean"),
        number_of_records=("canonical_smiles", "size")
    )
)

rows_after_duplicates = len(clean_df)

print("Rows before combining duplicates:", rows_before_duplicates)
print("Unique molecules remaining:", rows_after_duplicates)
print(
    "Duplicate records combined:",
    rows_before_duplicates - rows_after_duplicates
)

print(
    "Molecules that originally appeared more than once:",
    (clean_df["number_of_records"] > 1).sum()
)

clean_df.head()

In [ ]:
#docking-based classification labels
# Candidate for each individual target
clean_df["gsk3b_candidate"] = (
    clean_df["gsk3b_score"] <= DOCKING_CUTOFF
).astype(int)

clean_df["jnk3_candidate"] = (
    clean_df["jnk3_score"] <= DOCKING_CUTOFF
).astype(int)

# Dual candidate must pass the cutoff for both targets
clean_df["dual_candidate"] = (
    (clean_df["gsk3b_candidate"] == 1) &
    (clean_df["jnk3_candidate"] == 1)
).astype(int)

print("Classification labels created using cutoff:", DOCKING_CUTOFF)

In [ ]:
# Placing each molecule into one docking category

conditions = [
    (clean_df["gsk3b_candidate"] == 1) &
    (clean_df["jnk3_candidate"] == 1),

    (clean_df["gsk3b_candidate"] == 1) &
    (clean_df["jnk3_candidate"] == 0),

    (clean_df["gsk3b_candidate"] == 0) &
    (clean_df["jnk3_candidate"] == 1)
]

categories = [
    "Both targets",
    "GSK3B only",
    "JNK3 only"
]

clean_df["docking_category"] = np.select(
    conditions,
    categories,
    default="Neither target"
)

print(clean_df["docking_category"].value_counts())

In [ ]:
# Checking final cleaned dataset

final_check = pd.DataFrame({
    "Check": [
        "Final number of molecules",
        "Missing values",
        "Duplicate canonical SMILES",
        "Zero GSK3B scores",
        "Zero JNK3 scores"
    ],
    "Result": [
        len(clean_df),
        clean_df.isna().sum().sum(),
        clean_df.duplicated("canonical_smiles").sum(),
        (clean_df["gsk3b_score"] == 0).sum(),
        (clean_df["jnk3_score"] == 0).sum()
    ]
})

display(final_check)

print("\nDual-candidate counts:")
print(clean_df["dual_candidate"].value_counts())

print("\nDual-candidate percentages:")
print(
    clean_df["dual_candidate"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# save final processed dataset

final_columns = [
    "canonical_smiles",
    "smiles",
    "gsk3b_score",
    "jnk3_score",
    "number_of_records",
    "gsk3b_candidate",
    "jnk3_candidate",
    "dual_candidate",
    "docking_category"
]

clean_df = clean_df[final_columns].copy()

OUTPUT_FILE = "processed_molecules.csv"

clean_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Processed dataset saved successfully.")
print("File name:", OUTPUT_FILE)
print("Final dataset shape:", clean_df.shape)

clean_df.head()